# Fast DPO Alignment on Google Colab (T4 GPU)

Trains DPO on top of your **v1 fine-tuned model** in **~12-15 minutes** using Nvidia CUDA acceleration.

### Files to upload to Colab (left file pane, ~42 MB total):
1. `adapters.safetensors` (from `runs/finetune/adapters/adapters.safetensors`)
2. `train.jsonl` (from `data/dpo/train.jsonl`)

## 1. Install Dependencies
Installs pre-built wheels only (no compiling, takes ~15s).

In [ ]:
# Colab preinstalls broken torchao 0.10.0; remove it so PEFT uses standard PyTorch LoRA
!pip uninstall -y torchao
!pip install -q transformers peft accelerate bitsandbytes datasets trl safetensors gguf protobuf

## 2. Check Uploads & Convert MLX Adapter to PEFT

In [ ]:
import json, os, torch
from safetensors.torch import load_file, save_file

assert os.path.exists("adapters.safetensors"), "Upload adapters.safetensors to the Colab file browser!"
assert os.path.exists("train.jsonl"), "Upload train.jsonl to the Colab file browser!"

# Convert MLX weight matrices [in_dim, r] to PyTorch PEFT layout [r, in_dim]
print("Converting MLX adapter to PyTorch PEFT layout...")
mlx_weights = load_file("adapters.safetensors")
peft_dict = {}
for k, v in mlx_weights.items():
    if k == "__metadata__":
        continue
    new_k = "base_model.model." + k
    if new_k.endswith(".lora_a"):
        new_k = new_k[:-7] + ".lora_A.weight"
        peft_dict[new_k] = v.T.contiguous()
    elif new_k.endswith(".lora_b"):
        new_k = new_k[:-7] + ".lora_B.weight"
        peft_dict[new_k] = v.T.contiguous()
    else:
        peft_dict[new_k] = v

os.makedirs("v1_peft_adapter", exist_ok=True)
save_file(peft_dict, "v1_peft_adapter/adapter_model.safetensors")

peft_config = {
    "base_model_name_or_path": "Qwen/Qwen2.5-3B-Instruct",
    "bias": "none",
    "fan_in_fan_out": False,
    "inference_mode": True,
    "init_lora_weights": True,
    "layers_pattern": None,
    "layers_to_transform": None,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "modules_to_save": None,
    "peft_type": "LORA",
    "r": 16,
    "revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1",
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "task_type": "CAUSAL_LM"
}
with open("v1_peft_adapter/adapter_config.json", "w") as f:
    json.dump(peft_config, f, indent=2)

print("Adapter successfully prepared in ./v1_peft_adapter")

## 3. Load Base Model and Fuse v1 SFT Weights

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"

print("Downloading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Downloading base Qwen-3B...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=REVISION,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Attaching and merging v1 adapter...")
v1_model = PeftModel.from_pretrained(base_model, "./v1_peft_adapter")
v1_fused = v1_model.merge_and_unload()
print("Full v1 model assembled and ready in GPU VRAM!")

## 4. Load DPO Preference Dataset

In [ ]:
from datasets import Dataset

raw_data = []
with open("train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            raw_data.append(json.loads(line))

formatted_rows = []
for item in raw_data:
    prompt_str = tokenizer.apply_chat_template(
        item["messages"],
        tokenize=False,
        add_generation_prompt=True
    )
    formatted_rows.append({
        "prompt": prompt_str,
        "chosen": item["chosen"],
        "rejected": item["rejected"]
    })

dpo_dataset = Dataset.from_list(formatted_rows)
print(f"Loaded {len(dpo_dataset)} preference pairs.")

## 5. Run DPO Training (~12-15 mins on T4 GPU)

In [ ]:
import inspect
from trl import DPOTrainer, DPOConfig

dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

dpo_config_params = inspect.signature(DPOConfig.__init__).parameters
config_kwargs = {
    "output_dir": "./dpo_checkpoints",
    "learning_rate": 5e-6,
    "beta": 0.1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 1,
    "fp16": True,
    "logging_steps": 25,
    "save_strategy": "no",
    "remove_unused_columns": False,
    "report_to": "none",
}
if "max_length" in dpo_config_params:
    config_kwargs["max_length"] = 1024
if "max_prompt_length" in dpo_config_params:
    config_kwargs["max_prompt_length"] = 768

training_args = DPOConfig(**config_kwargs)

trainer_params = inspect.signature(DPOTrainer.__init__).parameters
trainer_kwargs = {
    "model": v1_fused,
    "ref_model": None,
    "args": training_args,
    "peft_config": dpo_lora_config,
    "train_dataset": dpo_dataset,
}
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

if "max_length" in trainer_params:
    trainer_kwargs["max_length"] = 1024
if "max_prompt_length" in trainer_params:
    trainer_kwargs["max_prompt_length"] = 768

trainer = DPOTrainer(**trainer_kwargs)

print("Starting CUDA-accelerated DPO training...")
trainer.train()
print("DPO training complete!")

## 6. Merge & Quantize to GGUF (for Ollama)

In [ ]:
print("Fusing DPO weights...")
final_model = trainer.model.merge_and_unload()
OUT = "qwen2.5-3b-haggle-dpo"
final_model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print(f"Saved fused HF model to ./{OUT}")

In [ ]:
%%bash
set -e
OUT="qwen2.5-3b-haggle-dpo"
[ -d llama.cpp ] || git clone -q --branch b6000 --depth 1 https://github.com/ggml-org/llama.cpp
python llama.cpp/convert_hf_to_gguf.py "$OUT" --outtype f16 --outfile "$OUT-f16.gguf"
cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
cmake --build llama.cpp/build --target llama-quantize -j > /dev/null
llama.cpp/build/bin/llama-quantize "$OUT-f16.gguf" "$OUT-q4_k_m.gguf" q4_k_m
rm "$OUT-f16.gguf"
echo "Export complete!"
ls -lh "$OUT-q4_k_m.gguf"

## 7. Download Model
Download `qwen2.5-3b-haggle-dpo-q4_k_m.gguf` (~1.9 GB) to your Mac.